# F1 Lap Time Prediction — LSTM Training Pipeline

**Visual Analytics Project** — Sapienza Università di Roma

This notebook demonstrates the LSTM model training pipeline for predicting F1 lap times and pit stop probability. The model takes a sliding window of the last 10 laps (12 features each) and predicts the next 5 lap times plus pit probability.

**Architecture**: LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(6)

**Training data**: Multi-season race data (2018-2025) from FastF1

## 1. Setup and Imports

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# Add backend to path
BACKEND_DIR = Path('.').resolve().parent
sys.path.insert(0, str(BACKEND_DIR))

# Paths
CACHE_DIR = BACKEND_DIR / 'data' / 'processed_cache'
MODEL_DIR = BACKEND_DIR / 'model' / 'saved'

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print(f'Backend dir: {BACKEND_DIR}')
print(f'Cache dir: {CACHE_DIR}')
print(f'Model dir: {MODEL_DIR}')

## 2. Data Discovery

We scan the processed cache directory for pre-computed session JSONs. Each file contains lap times, weather data, tyre strategy, and race results for one race.

In [ ]:
import re

# Discover all cached race sessions
pattern = re.compile(r'^session_(\d{4})_(\d+)_R\.json$')
sessions = []
for f in sorted(CACHE_DIR.iterdir()):
    m = pattern.match(f.name)
    if m:
        year, rnd = int(m.group(1)), int(m.group(2))
        sessions.append({'year': year, 'round': rnd, 'path': f})

sessions_df = pd.DataFrame(sessions)
print(f'Discovered {len(sessions_df)} race sessions')
print(f'Seasons: {sorted(sessions_df["year"].unique())}')
print(f'\nSessions per year:')
print(sessions_df.groupby('year').size().to_string())

## 3. Data Exploration

Let's load all laps and explore the dataset.

In [ ]:
# Load all laps from all sessions
all_laps = []
for _, row in sessions_df.iterrows():
    with open(row['path']) as f:
        data = json.load(f)
    for lap in data.get('lap_times', []):
        lap['_year'] = row['year']
        lap['_round'] = row['round']
        all_laps.append(lap)

laps_df = pd.DataFrame(all_laps)
print(f'Total laps: {len(laps_df):,}')
print(f'Drivers: {laps_df["Driver"].nunique()}')
print(f'\nLap time range: {laps_df["LapTimeSeconds"].min():.1f}s - {laps_df["LapTimeSeconds"].max():.1f}s')
print(f'Mean lap time: {laps_df["LapTimeSeconds"].mean():.1f}s')
print(f'\nCompound distribution:')
print(laps_df['Compound'].value_counts().to_string())

In [ ]:
# Lap Time Distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram of lap times (filter outliers)
valid_laps = laps_df[laps_df['LapTimeSeconds'].between(60, 150)]
axes[0].hist(valid_laps['LapTimeSeconds'], bins=60, color='#e10600', alpha=0.8, edgecolor='none')
axes[0].set_xlabel('Lap Time (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title('Lap Time Distribution')
axes[0].axvline(valid_laps['LapTimeSeconds'].median(), color='white', linestyle='--', alpha=0.5, label=f'Median: {valid_laps["LapTimeSeconds"].median():.1f}s')
axes[0].legend()

# Compound distribution
compound_colors = {'SOFT': '#e10600', 'MEDIUM': '#ffd700', 'HARD': '#ffffff', 'INTERMEDIATE': '#00e676', 'WET': '#2196f3'}
compound_counts = laps_df['Compound'].value_counts()
colors = [compound_colors.get(c, '#888888') for c in compound_counts.index]
axes[1].barh(compound_counts.index, compound_counts.values, color=colors, edgecolor='none')
axes[1].set_xlabel('Number of Laps')
axes[1].set_title('Compound Usage')

# Tyre life vs lap time
sample = valid_laps.sample(min(3000, len(valid_laps)), random_state=42)
scatter_colors = [compound_colors.get(c, '#888888') for c in sample['Compound']]
axes[2].scatter(sample['TyreLife'], sample['LapTimeSeconds'], c=scatter_colors, alpha=0.3, s=5)
axes[2].set_xlabel('Tyre Life (laps)')
axes[2].set_ylabel('Lap Time (seconds)')
axes[2].set_title('Tyre Degradation Effect')

plt.tight_layout()
plt.show()

In [ ]:
# Samples per season
laps_per_season = laps_df.groupby('_year').size()
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(laps_per_season.index.astype(str), laps_per_season.values, color='#e10600', edgecolor='none')
for bar, val in zip(bars, laps_per_season.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'{val:,}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Season')
ax.set_ylabel('Total Laps')
ax.set_title('Lap Counts per Season')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

The LSTM uses 12 features per lap:

| Feature | Description |
|---------|-------------|
| `LapTimeSeconds` | Lap time in seconds |
| `Sector1TimeSeconds` | Sector 1 split time |
| `Sector2TimeSeconds` | Sector 2 split time |
| `Sector3TimeSeconds` | Sector 3 split time |
| `compound_encoded` | Tyre compound (SOFT=0, MEDIUM=1, HARD=2, INTER=3, WET=4) |
| `TyreLife` | Number of laps on current set of tyres |
| `fuel_estimate` | Estimated fuel load (1.0 at start → 0.0 at end) |
| `track_temp` | Average track temperature (°C) |
| `Position` | Race position at end of lap |
| `GapToLeader` | Time gap to race leader (seconds) |
| `gap_ahead` | Gap change lap-to-lap |
| `drs_available` | DRS availability flag |

In [ ]:
# Build feature matrix for one session to show correlation
COMPOUND_MAP = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3, 'INTER': 3, 'WET': 4,
                'HYPERSOFT': 0, 'ULTRASOFT': 0, 'SUPERSOFT': 0, 'SUPERHARD': 2}

feat_cols = ['LapTimeSeconds', 'Sector1TimeSeconds', 'Sector2TimeSeconds', 'Sector3TimeSeconds',
             'compound_encoded', 'TyreLife', 'fuel_estimate', 'track_temp',
             'Position', 'GapToLeader', 'gap_ahead', 'drs_available']

# Use the full dataset for correlation
feat_df = valid_laps.copy()
feat_df['compound_encoded'] = feat_df['Compound'].map(COMPOUND_MAP).fillna(1)
total_laps_map = feat_df.groupby(['_year', '_round'])['LapNumber'].transform('max')
feat_df['fuel_estimate'] = (total_laps_map - feat_df['LapNumber'] + 1) / total_laps_map
feat_df['track_temp'] = 30.0  # placeholder for correlation
feat_df['GapToLeader'] = feat_df['GapToLeader'].fillna(0)
feat_df['gap_ahead'] = feat_df.groupby(['_year', '_round', 'Driver'])['GapToLeader'].diff().fillna(0)
feat_df['drs_available'] = 0

available = [c for c in feat_cols if c in feat_df.columns]
corr = feat_df[available].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. Sequence Building

We use a sliding window approach: **10 laps in → 5 lap times + pit probability out**.

Sequences with pit stops, outlier lap times (>150s), or NaN values are filtered out.

In [ ]:
from model.prepare_data import prepare_sequences, discover_all_rounds, SEQUENCE_LEN, PREDICT_LEN, LAP_FEATURES

# Discover rounds
rounds = discover_all_rounds()
print(f'Discovered {len(rounds)} races from {len(set(r[0] for r in rounds))} seasons')
print(f'Sources: {Counter(r[2] for r in rounds)}')
print(f'\nSequence config: {SEQUENCE_LEN} laps in → {PREDICT_LEN} predicted + pit prob')
print(f'Features per lap: {len(LAP_FEATURES)}')
for i, f in enumerate(LAP_FEATURES):
    print(f'  {i+1:2d}. {f}')

In [ ]:
# Load pre-built training data
X = np.load(MODEL_DIR / 'lstm_X.npy')
y = np.load(MODEL_DIR / 'lstm_y.npy')

with open(MODEL_DIR / 'data_stats.json') as f:
    stats = json.load(f)

print(f'X shape: {X.shape}  (samples, sequence_len, features)')
print(f'y shape: {y.shape}  (samples, pred_len + pit_prob)')
print(f'\ny_time_mean: {stats["y_time_mean"]:.3f}s')
print(f'y_time_std:  {stats["y_time_std"]:.3f}s')
print(f'\nPit probability distribution: {(y[:, -1] > 0).sum()} positive / {len(y)} total ({(y[:, -1] > 0).mean()*100:.1f}%)')

In [ ]:
# Visualize a sample sequence
idx = 42
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Input sequence — show all 12 features as heatmap
im = axes[0].imshow(X[idx].T, aspect='auto', cmap='viridis')
axes[0].set_xlabel('Lap in sequence')
axes[0].set_ylabel('Feature')
axes[0].set_yticks(range(len(LAP_FEATURES)))
axes[0].set_yticklabels(LAP_FEATURES, fontsize=8)
axes[0].set_title(f'Input Sequence (sample #{idx}, normalized)')
plt.colorbar(im, ax=axes[0])

# Target — denormalize and show
y_denorm = y[idx, :5] * stats['y_time_std'] + stats['y_time_mean']
pit = y[idx, 5]
axes[1].bar(range(1, 6), y_denorm, color='#e10600', edgecolor='none')
axes[1].set_xlabel('Predicted Lap')
axes[1].set_ylabel('Lap Time (seconds)')
axes[1].set_title(f'Target: Next 5 Laps (pit prob: {pit:.2f})')
axes[1].set_xticks(range(1, 6))

plt.tight_layout()
plt.show()

## 6. Train/Validation Split

In [ ]:
# 80/20 split (same as train.py)
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(len(X) * 0.8)

X_train, X_val = X[indices[:split]], X[indices[split:]]
y_train, y_val = y[indices[:split]], y[indices[split:]]

print(f'Training set:   {X_train.shape[0]:,} samples')
print(f'Validation set: {X_val.shape[0]:,} samples')
print(f'Split ratio: {X_train.shape[0]/len(X)*100:.0f}% / {X_val.shape[0]/len(X)*100:.0f}%')

## 7. Model Architecture

The F1LapPredictor is a stacked LSTM with dropout regularization:

```
Input (10, 12) → LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(6)
                                                                         ↓
                                                      [5 lap times + 1 pit probability]
```

Loss function: MSE (lap times) + 2× BCE (pit probability)

In [ ]:
import torch
import torch.nn as nn
from model.train import F1LapPredictor, combined_loss

model = F1LapPredictor(n_features=12, hidden1=64, hidden2=32, pred_len=6)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 8. Training

Training with Adam optimizer, ReduceLROnPlateau scheduler, early stopping (patience=15), and gradient clipping.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import time

# Replace NaN/inf
X_train_clean = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
y_train_clean = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val_clean = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
y_val_clean = np.nan_to_num(y_val, nan=0.0, posinf=0.0, neginf=0.0)

train_ds = TensorDataset(torch.tensor(X_train_clean, dtype=torch.float32),
                         torch.tensor(y_train_clean, dtype=torch.float32))
val_ds = TensorDataset(torch.tensor(X_val_clean, dtype=torch.float32),
                       torch.tensor(y_val_clean, dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

model = F1LapPredictor(n_features=12, hidden1=64, hidden2=32, pred_len=6)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

train_losses, val_losses, lrs = [], [], []
best_val_loss = float('inf')
patience_counter = 0
max_epochs = 100

start_time = time.time()

for epoch in range(max_epochs):
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = combined_loss(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            pred = model(X_batch)
            loss = combined_loss(pred, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(val_ds)

    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    lrs.append(lr)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= 15:
            print(f'Early stopping at epoch {epoch+1}')
            break

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{max_epochs} — train: {train_loss:.4f}, val: {val_loss:.4f}, lr: {lr:.6f}')

elapsed = time.time() - start_time
print(f'\nTraining completed in {elapsed:.1f}s ({len(train_losses)} epochs)')
print(f'Best val_loss: {best_val_loss:.4f}')

# Restore best weights
model.load_state_dict(best_state)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(train_losses) + 1)

axes[0].plot(epochs_range, train_losses, label='Train Loss', color='#e10600', alpha=0.8)
axes[0].plot(epochs_range, val_losses, label='Val Loss', color='#00e676', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Combined Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(epochs_range, lrs, color='#ffd700')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].set_yscale('log')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 9. Evaluation

In [ ]:
# Predict on validation set
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        pred = model(X_batch)
        all_preds.append(pred.numpy())
        all_targets.append(y_batch.numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)

# Denormalize lap times (first 5 columns)
pred_times = preds[:, :5] * stats['y_time_std'] + stats['y_time_mean']
true_times = targets[:, :5] * stats['y_time_std'] + stats['y_time_mean']

# Metrics
mae = np.abs(pred_times - true_times).mean()
rmse = np.sqrt(((pred_times - true_times) ** 2).mean())
per_step_mae = np.abs(pred_times - true_times).mean(axis=0)

# Pit probability accuracy
pit_pred = (torch.sigmoid(torch.tensor(preds[:, 5])) > 0.5).numpy()
pit_true = targets[:, 5] > 0.5
pit_acc = (pit_pred == pit_true).mean()

print(f'=== Evaluation on {len(targets):,} validation samples ===')
print(f'\nLap Time Prediction:')
print(f'  MAE:  {mae:.3f}s')
print(f'  RMSE: {rmse:.3f}s')
print(f'\n  Per-step MAE:')
for i, m in enumerate(per_step_mae):
    print(f'    Lap +{i+1}: {m:.3f}s')
print(f'\nPit Stop Prediction:')
print(f'  Accuracy: {pit_acc*100:.1f}%')
print(f'  True positives: {(pit_pred & pit_true).sum()} / {pit_true.sum()}')

In [ ]:
# Predicted vs Actual scatter + Residual distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter — predicted vs actual (sample for clarity)
sample_idx = np.random.choice(len(pred_times), min(2000, len(pred_times)), replace=False)
p_flat = pred_times[sample_idx].flatten()
t_flat = true_times[sample_idx].flatten()
axes[0].scatter(t_flat, p_flat, alpha=0.15, s=3, color='#e10600')
lims = [min(t_flat.min(), p_flat.min()), max(t_flat.max(), p_flat.max())]
axes[0].plot(lims, lims, '--', color='white', alpha=0.5, label='Perfect')
axes[0].set_xlabel('Actual Lap Time (s)')
axes[0].set_ylabel('Predicted Lap Time (s)')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()
axes[0].set_aspect('equal')

# Residual distribution
residuals = (pred_times - true_times).flatten()
axes[1].hist(residuals, bins=80, color='#2196f3', alpha=0.8, edgecolor='none')
axes[1].axvline(0, color='white', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Residual (s)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residuals (mean={residuals.mean():.3f}s, std={residuals.std():.3f}s)')

# Per-step MAE bar chart
axes[2].bar(range(1, 6), per_step_mae, color='#ffd700', edgecolor='none')
axes[2].set_xlabel('Prediction Step')
axes[2].set_ylabel('MAE (seconds)')
axes[2].set_title('MAE by Prediction Horizon')
axes[2].set_xticks(range(1, 6))
axes[2].set_xticklabels([f'Lap +{i}' for i in range(1, 6)])

plt.tight_layout()
plt.show()

## 10. Example Predictions

Let's visualize predictions for a specific driver in a specific race.

In [ ]:
# Pick a sample from validation and show actual vs predicted
n_examples = 4
fig, axes = plt.subplots(1, n_examples, figsize=(16, 4))

example_indices = np.random.choice(len(pred_times), n_examples, replace=False)

for ax, idx in zip(axes, example_indices):
    x = range(1, 6)
    ax.plot(x, true_times[idx], 'o-', color='white', label='Actual', markersize=6)
    ax.plot(x, pred_times[idx], 's--', color='#e10600', label='Predicted', markersize=6)
    ax.set_xlabel('Lap +N')
    ax.set_ylabel('Lap Time (s)')
    ax.set_title(f'Sample #{idx}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xticks(x)
    ax.grid(alpha=0.2)

plt.suptitle('Example Predictions: Actual vs Predicted Lap Times', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Conclusion

### Summary

| Metric | Value |
|--------|-------|
| Training samples | See output above |
| Validation samples | See output above |
| Races used | See output above |
| Model parameters | 32,326 |
| Best validation loss | See output above |
| Lap time MAE | See output above |
| Lap time RMSE | See output above |
| Pit stop accuracy | See output above |

### Key Findings

1. **Lightweight model**: Only 32K parameters — trains in seconds on CPU, no GPU required
2. **Multi-season generalization**: Model trained across different circuits, weather conditions, and regulation eras
3. **Prediction quality degrades with horizon**: Near-term predictions (Lap +1) are more accurate than far-term (Lap +5), as expected
4. **Pit stop detection**: Binary classification via sigmoid on the 6th output; weighted 2x in the loss

### Expanding the Dataset

To download more seasons and retrain:
```bash
cd backend
python -m scripts.bulk_download             # Download all 2018-2025 data
python -m model.prepare_data                # Rebuild training sequences
python -m model.train                       # Retrain LSTM
```